# Week 2 Lab — HOG Features + Simple Classifier
Run cells top to bottom. Step 1 below: upload `week2_hog_data.zip` when prompted.

In [ ]:
!pip install -q scikit-image scikit-learn opencv-python-headless

In [ ]:
from google.colab import files
uploaded = files.upload()  # choose week2_hog_data.zip from your computer

In [ ]:
import zipfile, os

with zipfile.ZipFile('week2_hog_data.zip') as z:
    z.extractall('.')

print('images:', os.listdir('images'))
print('dataset classes:', os.listdir('dataset'))
for c in os.listdir('dataset'):
    print(' ', c, '->', len(os.listdir(os.path.join('dataset', c))), 'images')

## Part 1 — Image Gradients

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread('images/sample1.jpg', cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (256, 256))

gx = cv2.Sobel(img, cv2.CV_32F, 1, 0, ksize=1)
gy = cv2.Sobel(img, cv2.CV_32F, 0, 1, ksize=1)

magnitude, angle = cv2.cartToPolar(gx, gy, angleInDegrees=True)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Original')
axes[1].imshow(magnitude, cmap='gray'); axes[1].set_title('Gradient Magnitude')
axes[2].imshow(angle, cmap='hsv'); axes[2].set_title('Gradient Direction')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('gradients_output.png', dpi=150)
plt.show()

**Reflection Question 1:** Look at the gradient magnitude image. Which parts of the photo appear brightest, and why do edges and object boundaries tend to produce strong gradients while flat, uniform regions (like a clear sky) do not?

## Part 2 — Extracting HOG Features

In [ ]:
from skimage.feature import hog
from skimage import exposure

img = cv2.imread('images/sample1.jpg', cv2.IMREAD_GRAYSCALE)
img = cv2.resize(img, (128, 128))

features, hog_image = hog(
    img,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm='L2-Hys',
    visualize=True,
)
print('HOG feature vector shape:', features.shape)

hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img, cmap='gray'); axes[0].set_title('Input Image')
axes[1].imshow(hog_image_rescaled, cmap='gray'); axes[1].set_title('HOG Visualization')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('hog_output.png', dpi=150)
plt.show()

**Reflection Question 2:** If you increase `pixels_per_cell` from (8, 8) to (16, 16), do you expect the resulting feature vector to get longer or shorter, and what does that mean for the level of spatial detail captured?

## Part 4 — Training a Simple Classifier on HOG Features
Uses the `dataset/` folder (currently has `cats`, `dogs`, `birds`). The lab itself expects two classes — you can leave all three in for now (it will still run as a 3-class problem, which is actually what the Week 2 assignment wants), or remove one folder if you want to match the lab exactly first.

In [ ]:
import os
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

DATASET_DIR = 'dataset'
IMG_SIZE = (128, 128)

def extract_hog_features(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, IMG_SIZE)
    return hog(
        img, orientations=9, pixels_per_cell=(8, 8),
        cells_per_block=(2, 2), block_norm='L2-Hys',
    )

X, y = [], []
class_names = sorted(os.listdir(DATASET_DIR))
for label_idx, class_name in enumerate(class_names):
    class_folder = os.path.join(DATASET_DIR, class_name)
    for filename in os.listdir(class_folder):
        path = os.path.join(class_folder, filename)
        try:
            X.append(extract_hog_features(path))
            y.append(label_idx)
        except Exception as e:
            print(f'Skipping {path}: {e}')

X = np.array(X)
y = np.array(y)
print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features each')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf = LinearSVC(max_iter=10000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print('\nAccuracy:', accuracy_score(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=class_names))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

**Reflection Question 3:** Look at your confusion matrix. Are misclassifications spread evenly, or concentrated between specific pairs of categories? What visual similarities between those categories might explain the confusion?